### Step 1: Imports and Basing Preprocessing:

In [3]:
import pandas as pd
from sklearn.svm import SVR
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import optuna
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import mlflow
import mlflow.sklearn
import dagshub
import json
# from catboost import CatBoostRegressor

D:\Andacoda\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
D:\Andacoda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [4]:
mlflow.set_tracking_uri("https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow")

dagshub.init(repo_owner='PriyanshuMewal', repo_name='delivery-time-prediction', mlflow=True)

mlflow.set_experiment("Experiment 4: Model Selection Without Missing Values")

Accessing as PriyanshuMewal

Initialized MLflow to track repo "PriyanshuMewal/delivery-time-prediction"

Repository PriyanshuMewal/delivery-time-prediction initialized!

<Experiment: artifact_location='mlflow-artifacts:/0159417f09e1476798193b9381e1c2c4', creation_time=1771566150176, experiment_id='4', last_update_time=1771566150176, lifecycle_stage='active', name='Experiment 4: Model Selection Without Missing Values', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [5]:
train_df = pd.read_csv("../data/processed/without_missing_values/train_df.csv")
test_df = pd.read_csv("../data/processed/without_missing_values/test_df.csv")

In [8]:
train_df.head()

,Unnamed: 0,ord_cat__traffic,ord_cat__distance_type,ord_cat__order_time_of_day,nom_cat__weather_fog,nom_cat__weather_sandstorms,nom_cat__weather_stormy,nom_cat__weather_sunny,nom_cat__weather_windy,nom_cat__order_type_drinks,...,nom_cat__city_type_urban,num__age,num__ratings,num__pickup_time,num__distance,remainder__condition,remainder__multi_deliveries,remainder__order_month,remainder__is_weekend,time
0,2788,0.0,1.0,3.0,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.473684,0.84,0.714765,0.404182,1,0.0,4,1,-1.036137
1,36411,2.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.631579,0.88,0.740772,0.243678,1,0.0,3,1,0.138373
2,32533,3.0,3.0,2.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,1.000000,0.64,0.573826,0.787364,0,0.0,4,0,1.526431
3,29327,2.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.263158,0.88,0.660235,0.236691,0,0.0,4,1,-0.609042
4,40988,1.0,1.0,2.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.052632,0.92,0.360738,0.397584,0,1.0,3,1,-0.609042


In [9]:
print(train_df.duplicated().sum())
test_df.duplicated().sum()

0


0

In [10]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29532 entries, 0 to 29531
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Unnamed: 0                        29532 non-null  int64  
 1   ord_cat__traffic                  29532 non-null  float64
 2   ord_cat__distance_type            29532 non-null  float64
 3   ord_cat__order_time_of_day        29532 non-null  float64
 4   nom_cat__weather_fog              29532 non-null  float64
 5   nom_cat__weather_sandstorms       29532 non-null  float64
 6   nom_cat__weather_stormy           29532 non-null  float64
 7   nom_cat__weather_sunny            29532 non-null  float64
 8   nom_cat__weather_windy            29532 non-null  float64
 9   nom_cat__order_type_drinks        29532 non-null  float64
 10  nom_cat__order_type_meal          29532 non-null  float64
 11  nom_cat__order_type_snack         29532 non-null  float64
 12  nom_cat__vehicl

In [11]:
test_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7383 entries, 0 to 7382
Data columns (total 26 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Unnamed: 0                        7383 non-null   int64  
 1   ord_cat__traffic                  7383 non-null   float64
 2   ord_cat__distance_type            7383 non-null   float64
 3   ord_cat__order_time_of_day        7383 non-null   float64
 4   nom_cat__weather_fog              7383 non-null   float64
 5   nom_cat__weather_sandstorms       7383 non-null   float64
 6   nom_cat__weather_stormy           7383 non-null   float64
 7   nom_cat__weather_sunny            7383 non-null   float64
 8   nom_cat__weather_windy            7383 non-null   float64
 9   nom_cat__order_type_drinks        7383 non-null   float64
 10  nom_cat__order_type_meal          7383 non-null   float64
 11  nom_cat__order_type_snack         7383 non-null   float64
 12  nom_cat__vehicle_

In [6]:
X_train = train_df.drop(columns=["time"])
X_test = test_df.drop(columns=["time"])

y_train = train_df["time"]
y_test = test_df["time"]

### Experimentation: Hyperparameter Tunning Over XGBoost, LightGBM, Catboost, Random Forest and SVM:

In [17]:
def objactive(trial, model):

    model_name = model().__class__.__name__

    param_dict = {}

    if model_name in [XGBRegressor().__class__.__name__, LGBMRegressor().__class__.__name__]:

        param_dict["n_estimators"] = trial.suggest_int("n_estimators", 50, 300)
        param_dict["max_depth"] = trial.suggest_int("max_depth", 3, 10)
        param_dict["learning_rate"] = trial.suggest_float("learning_rate", 1e-3, 1e-1, log=True)

    elif model_name == RandomForestRegressor().__class__.__name__:

        param_dict["n_estimators"] = trial.suggest_int("n_estimators", 50, 300)
        param_dict["max_depth"] = trial.suggest_int("max_depth", 3, 10)
        param_dict["min_samples_split"] = trial.suggest_int("min_samples_split", 2, 20)
        param_dict["min_samples_leaf"] = trial.suggest_int("min_samples_leaf", 1, 20)

    elif model_name == SVR().__class__.__name__:

        param_dict["C"] = trial.suggest_float("C", 0.1, 100, log=True)
        param_dict["epsilon"] = trial.suggest_float("epsilon", 0.005, 0.3)
        param_dict["gamma"] = trial.suggest_float("gamma", 1e-4, 1.0, log=True)

    if model_name == XGBRegressor().__class__.__name__:

        param_dict["tree_method"] = "hist"
        param_dict["device"] = "cuda"

    elif model_name == LGBMRegressor().__class__.__name__:

        param_dict["device"] = "gpu"
        param_dict["gpu_platform_id"] = 0
        param_dict["gpu_device_id"] = 0

    # elif model_name == CatBoostRegressor().__class__.__name__:

    #     param_dict.update({"loss_function": "RMSE",
    #                       "iterations": trial.suggest_int("iterations", 500, 4000),
    #                       "learning_rate": trial.suggest_float("learning_rate", 1e-2, 1e-1, log=True),
    #                       "depth": trial.suggest_int("depth", 5, 9),
    #                       "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 3, 10),
    #                       "task_type": "GPU",
    #                        "verbose": 0})


    model = model(**param_dict)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_true=y_test, y_pred=y_pred)
    r2 = r2_score(y_test, y_pred)

    trial.set_user_attr("r2", r2)
    return mae

In [12]:
best_params = {}

# Hyperparameter tunning over XGboost:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, XGBRegressor), n_trials=30)

best_params.update({"Xgboost": study.best_params})

[I 2026-02-19 14:29:13,801] A new study created in memory with name: no-name-8d7f68b1-ad2f-4c2d-b2d8-8afd6b92809b
/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [14:29:14] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
[I 2026-02-19 14:29:14,956] Trial 0 finished with value: 0.48100724827849883 and parameters: {'n_estimators': 154, 'max_depth': 6, 'learning_rate': 0.0072492608918719465}. Best is trial 0 with value: 0.48100724827849883.
[I 2026-02-19 14:29:15,217] Trial 1 finished with value: 0.3402886187360073 and parameters: {'n_estimators': 5

In [13]:
# Hyperparameter tunning over LightGBM:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, LGBMRegressor), n_trials=30)

best_params.update({"lightgbm": study.best_params})

[I 2026-02-19 14:30:22,906] A new study created in memory with name: no-name-f4b64fef-323d-41a9-b2e7-1785e58da2a9


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 727
[LightGBM] [Info] Number of data points in the train set: 33472, number of used features: 26
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 12 dense feature groups (0.38 MB) transferred to GPU in 0.001041 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 0.000000


[I 2026-02-19 14:30:30,158] Trial 0 finished with value: 0.6783050157076249 and parameters: {'n_estimators': 206, 'max_depth': 9, 'learning_rate': 0.0012481503570990035}. Best is trial 0 with value: 0.6783050157076249.


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 727
[LightGBM] [Info] Number of data points in the train set: 33472, number of used features: 26
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 12 dense feature groups (0.38 MB) transferred to GPU in 0.000980 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 0.000000


[I 2026-02-19 14:30:31,851] Trial 1 finished with value: 0.5161317254571376 and parameters: {'n_estimators': 280, 'max_depth': 9, 'learning_rate': 0.002903520903877274}. Best is trial 1 with value: 0.5161317254571376.


...

[I 2026-02-19 14:31:05,735] Trial 27 finished with value: 0.34169301319792533 and parameters: {'n_estimators': 176, 'max_depth': 9, 'learning_rate': 0.04669272074611471}. Best is trial 21 with value: 0.3358374709166689.


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 727
[LightGBM] [Info] Number of data points in the train set: 33472, number of used features: 26
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 12 dense feature groups (0.38 MB) transferred to GPU in 0.001155 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


[I 2026-02-19 14:31:07,064] Trial 28 finished with value: 0.3443735105172215 and parameters: {'n_estimators': 258, 'max_depth': 5, 'learning_rate': 0.0982166245952981}. Best is trial 21 with value: 0.3358374709166689.


[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 727
[LightGBM] [Info] Number of data points in the train set: 33472, number of used features: 26
[LightGBM] [Info] Using requested OpenCL platform 0 device 0
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 12 dense feature groups (0.38 MB) transferred to GPU in 0.000992 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score 0.000000


[I 2026-02-19 14:31:08,316] Trial 29 finished with value: 0.33762965023745006 and parameters: {'n_estimators': 222, 'max_depth': 9, 'learning_rate': 0.07895592552260403}. Best is trial 21 with value: 0.3358374709166689.


In [ ]:
# Hyperparameter tunning over RandomForestRegressor:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, RandomForestRegressor), n_trials=30)

best_params.update({"random_forest": study.best_params})

[I 2026-02-18 12:13:55,682] A new study created in memory with name: no-name-d094d886-6060-4b97-bda6-37ea74315353
[I 2026-02-18 12:14:27,804] Trial 0 finished with value: 3.5136020344847756 and parameters: {'n_estimators': 245, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 3}. Best is trial 0 with value: 3.5136020344847756.
[I 2026-02-18 12:14:42,214] Trial 1 finished with value: 4.624289482878035 and parameters: {'n_estimators': 270, 'max_depth': 5, 'min_samples_split': 15, 'min_samples_leaf': 13}. Best is trial 0 with value: 3.5136020344847756.
[I 2026-02-18 12:15:09,487] Trial 2 finished with value: 3.5342995613409625 and parameters: {'n_estimators': 198, 'max_depth': 10, 'min_samples_split': 13, 'min_samples_leaf': 13}. Best is trial 0 with value: 3.5136020344847756.
[I 2026-02-18 12:15:30,797] Trial 3 finished with value: 5.166019592055856 and parameters: {'n_estimators': 297, 'max_depth': 4, 'min_samples_split': 20, 'min_samples_leaf': 19}. Best is trial 0 with va

In [15]:
# Hyperparameter tunning over Catboost:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, CatBoostRegressor), n_trials=30)

best_params.update({"catboost": study.best_params})

[I 2026-02-19 14:35:16,504] A new study created in memory with name: no-name-fdb4d577-b41c-4496-831b-4685bddbc809
[I 2026-02-19 14:35:51,304] Trial 0 finished with value: 0.3296936694054538 and parameters: {'iterations': 2337, 'learning_rate': 0.08229612607147098, 'depth': 9, 'l2_leaf_reg': 5.987961407928152}. Best is trial 0 with value: 0.3296936694054538.
[I 2026-02-19 14:36:07,414] Trial 1 finished with value: 0.3437155391606496 and parameters: {'iterations': 996, 'learning_rate': 0.015496021890583826, 'depth': 8, 'l2_leaf_reg': 3.1911457641454946}. Best is trial 0 with value: 0.3296936694054538.
[I 2026-02-19 14:36:32,545] Trial 2 finished with value: 0.35047591956666146 and parameters: {'iterations': 2813, 'learning_rate': 0.012257297606799894, 'depth': 5, 'l2_leaf_reg': 7.7704317310793485}. Best is trial 0 with value: 0.3296936694054538.
[I 2026-02-19 14:37:06,609] Trial 3 finished with value: 0.33994703593998593 and parameters: {'iterations': 3605, 'learning_rate': 0.01947220877

In [ ]:
# Hyperparameter tunning over SVM:
study = optuna.create_study(direction="minimize")
study.optimize(lambda trial: objactive(trial, SVR), n_trials=15)

best_params.update({"svm": study.best_params})

[I 2026-02-20 11:55:03,832] A new study created in memory with name: no-name-6ce542b5-055d-4bad-94e4-bed707d6c188
[I 2026-02-20 12:00:47,425] Trial 0 finished with value: 0.38830654432263245 and parameters: {'C': 40.98418430811215, 'epsilon': 0.19698372343974443, 'gamma': 0.03367119148318814}. Best is trial 0 with value: 0.38830654432263245.
[I 2026-02-20 12:02:28,013] Trial 1 finished with value: 0.4443171107057237 and parameters: {'C': 0.30420649599479316, 'epsilon': 0.1583459632732593, 'gamma': 0.018625674823634336}. Best is trial 0 with value: 0.38830654432263245.
[I 2026-02-20 12:04:24,144] Trial 2 finished with value: 0.5090590985578147 and parameters: {'C': 0.1077727309592721, 'epsilon': 0.07048023356791441, 'gamma': 0.00656859863851054}. Best is trial 0 with value: 0.38830654432263245.
[I 2026-02-20 12:08:46,788] Trial 3 finished with value: 0.38154004225511257 and parameters: {'C': 0.6402362023526725, 'epsilon': 0.09578490563350647, 'gamma': 0.3202161385892054}. Best is trial 

In [8]:
print(best_params)

{'Xgboost': {'n_estimators': 249, 'max_depth': 10, 'learning_rate': 0.020719383276707765}, 'lightgbm': {'n_estimators': 236, 'max_depth': 10, 'learning_rate': 0.08189043025104488}, 'catboost': {'iterations': 3974, 'learning_rate': 0.04868902023272203, 'depth': 9, 'l2_leaf_reg': 3.27486722555657}}


### Logging Every Experiment in Mlflow:

In [8]:
def log_mlflow(model, X_train, y_train, X_test, y_test):

    with mlflow.start_run(run_name=model.__class__.__name__):

        parameters = {"model": model.__class__.__name__}

        # logging parameters:
        parameters.update(model.get_params())
        mlflow.log_params(parameters)

        # Train and evaluate model:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        # logging metrics:
        mlflow.log_metric("mean_absolute_error", mae)
        mlflow.log_metric("r2_score", r2)

        # log model:
        # signature = mlflow.models.infer_signature(X_test, y_pred)
        # mlflow.sklearn.log_model(model, signature=signature)

        # logging datasets:
        # mlflow.log_artifact("../data/interim/reddit_cleaned.csv")

        # Set tags:
        # mlflow.set_tag("Author", "Priyanshu Mewal")
        # mlflow.set_tag("Description", "Food delivery prediction using Iterative Imputer with max_iter=15.")
        # mlflow.set_tag("Experiment", "HP over iterative imputer")

In [9]:
for model_name, params in best_params.items():

    if model_name == "Xgboost":
       model = XGBRegressor(**params)
    elif model_name == "lightgbm":
        model = LGBMRegressor(**params)
    elif model_name == "random_forest":
        model = RandomForestRegressor(**params)
    elif model_name == "catboost":
        model = CatBoostRegressor(**params)
    else:
        model = SVR(**params)

    log_mlflow(model, X_train, y_train, X_test, y_test)

🏃 View run SVR at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/4/runs/57a4f6b9534942a39ded060306e4b122
🧪 View experiment at: https://dagshub.com/PriyanshuMewal/delivery-time-prediction.mlflow/#/experiments/4
